<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/multiomics/notebooks/01_multiomics_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔀 Multi-omics I — Integrating the proteome and the metabolome

---

For two and a half days we have treated our cohort as two separate experiments. They are
not. The **same 45 serum samples** were split in two and sent to two instruments. Every
patient therefore has two descriptions of the same underlying physiology, and this session
is about using both at once.

### What you will be able to do afterwards

1. Say what "integration" actually means, and pick the right kind for a given question.
2. Prepare two omics matrices so they *can* be integrated — the unglamorous step where
   most integration attempts fail.
3. Run **Similarity Network Fusion (SNF)** and **Multi-Omics Factor Analysis (MOFA)** and
   interpret their output.
4. Tell a factor that captures biology from one that captures a technical artefact.

## 🧬 Why bother?

Integration is fashionable, which is a reason to be sceptical. It earns its keep when it
answers a question that neither layer can answer alone. There are three such questions.

**1. Is the signal shared or complementary?**
If proteins and metabolites move together, they are reporting the same process, and
measuring both mainly buys you confidence. If they move independently, each is telling you
something the other cannot — and *that* is when integration adds information rather than
reassurance.

**2. Can we stratify patients better?**
Neither the proteome nor the metabolome separated our three groups cleanly. A weak signal
in two layers can add up to a usable one — or can fail to, which is also worth knowing.

**3. Where does the mechanism cross layers?**
This is the real payoff. An enzyme is a protein; its substrate and product are metabolites.
A change in one without the other means something different from a change in both. The
published analysis of this cohort found exactly such a crossing: **cysteine and methionine
metabolism**, linked through **MAT2B**, the regulatory subunit of methionine
adenosyltransferase. Neither layer alone points there. We chase that down in Multi-omics II.

### ⚙️ A short taxonomy

| Strategy | How it works | Example methods | When to use it |
|---|---|---|---|
| **Early** (concatenation) | glue the matrices side by side, analyse as one | PCA, penalised regression on the joint matrix | rarely a good idea: the layer with more features and larger variance dominates |
| **Intermediate** (joint modelling) | learn a shared latent space across layers | **MOFA**, MCIA, JIVE, DIABLO, **SNF** | the default for discovery — this notebook |
| **Late** (meta-analysis) | analyse each layer separately, combine the results | joint pathway analysis, rank aggregation | interpretable and robust; Multi-omics II |

Cutting the same cake a different way: **unsupervised** methods (MOFA, SNF) ignore the
group labels and ask what structure exists; **supervised** methods (DIABLO, sPLS-DA) use
the labels to find structure that separates them. With 15 patients per group, supervised
methods overfit spectacularly, so we stay unsupervised and use the labels only to *check*
what we found. That discipline is the difference between a finding and a delusion.

In [ ]:
%pip install -q snfpy mofapy2 mofax

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

OUT_DIR = Path("multiomics/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
GROUP_COLOURS = {"Con": "#4C72B0", "CSKP": "#DD8452", "CRKP": "#C44E52"}
GROUP_ORDER = ["Con", "CSKP", "CRKP"]

## 1. Building the two views

An integration method needs the two matrices to share **the same samples, in the same
order**. That sounds trivial. It is where things break.

In [ ]:
proteins_raw = pd.read_csv(f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv", sep="\t")
metabolites_raw = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t").set_index("sample_id")
metabolite_annotation = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv", sep="\t")

protein_info = proteins_raw[["protein_group", "genes", "description"]].set_index("protein_group")
print(f"proteomics : {proteins_raw.shape[0]} protein groups")
print(f"metabolomics: {metabolites_raw.shape[0]} metabolites")

### The join, done carefully

Both repositories name the patients, but not identically — the proteomics acquisition
software wrote `CPKP` where everything else writes `CRKP`, and each platform has its own
pooled quality-control samples (`QC_pool1..3` vs `QC01..06`) which are **not** patients.
We therefore intersect on the clinical sample identifiers and check the result, rather
than assuming the columns line up.

In [ ]:
protein_samples = [c for c in proteins_raw.columns if c in metadata.index]
metabolite_samples = [c for c in metabolites_raw.columns if c in metadata.index]
shared = [s for s in metadata.index if s in protein_samples and s in metabolite_samples]

print(f"patients with proteomics    : {len(protein_samples)}")
print(f"patients with metabolomics  : {len(metabolite_samples)}")
print(f"patients with both          : {len(shared)}")
assert len(shared) == 45, "the two layers no longer describe the same cohort — stop and find out why"

groups = metadata.loc[shared, "group"]
groups.value_counts().reindex(GROUP_ORDER).to_frame("patients")

> 💡 That `assert` is not decoration. Silent sample mismatches are the single most common
> way a multi-omics analysis produces confident nonsense: the matrices still have the right
> shape, the code still runs, and every result is wrong. Assert what you believe.

### Preparing each view

The two layers are on incomparable scales (MaxLFQ intensities vs MRM peak areas), have
different missingness (27 % vs 5 %) and different numbers of features. We apply the same
recipe to both, deliberately keeping it simple and symmetric:

1. log₂ transform — both are multiplicative measurements
2. drop features missing in more than 30 % of patients
3. impute the rest with the feature minimum (missing = low)
4. **z-score each feature** — so that no feature dominates because of its units, and no
   layer dominates because of its dynamic range

In [ ]:
def prepare_view(wide: pd.DataFrame, feature_col: str, samples: list[str],
                 max_missing: float = 0.30) -> pd.DataFrame:
    """Return a samples x features matrix: log2, filtered, imputed and z-scored."""
    matrix = wide.set_index(feature_col)[samples].T          # samples in rows
    matrix = np.log2(matrix)
    keep = matrix.isna().mean() <= max_missing
    matrix = matrix.loc[:, keep]
    matrix = matrix.fillna(matrix.min())                      # missing = below detection
    matrix = (matrix - matrix.mean()) / matrix.std()          # z-score per feature
    return matrix.loc[:, matrix.std() > 0]


proteomics = prepare_view(proteins_raw, "protein_group", shared)
metabolomics = prepare_view(metabolites_raw, "metabolite", shared)

views = {"proteomics": proteomics, "metabolomics": metabolomics}
pd.DataFrame(
    {name: {"samples": v.shape[0], "features": v.shape[1]} for name, v in views.items()}
).T

In [ ]:
import unicodedata

def ascii_safe(names):
    """Transliterate names to pure ASCII."""
    out = []
    for name in names:
        name = str(name)
        name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
        out.append(name.strip())
    return out


for view_name, view_df in views.items():
    bad = [c for c in view_df.columns if not str(c).isascii()]
    if bad:
        print(f"{view_name}: {len(bad)} non-ASCII feature name(s), e.g. {bad[:5]}")
        view_df.columns = ascii_safe(view_df.columns)


⚠️ Note the feature imbalance. Whatever the exact numbers, the two views will not have the
same width, and **early integration would let the wider one shout down the other**. Both
methods below handle this properly — SNF because it converts each view to a
sample × sample similarity before combining, MOFA because it models each view's variance
separately. That is the main reason to prefer them over concatenation.

## 2. Do the two layers agree at all?

Before any model, one honest question: do patients who look similar in the proteome also
look similar in the metabolome? We can answer it with the sample-to-sample distance
matrices and a **Mantel test** — correlate the two distance matrices, then permute the
sample labels to see how big that correlation could be by chance.

In [ ]:
from scipy.spatial.distance import pdist, squareform

d_prot = pdist(proteomics.values, metric="euclidean")
d_metab = pdist(metabolomics.values, metric="euclidean")
observed = stats.spearmanr(d_prot, d_metab).statistic

rng = np.random.default_rng(42)
null = []
square_metab = squareform(d_metab)
for _ in range(999):
    order = rng.permutation(len(shared))
    null.append(stats.spearmanr(d_prot, squareform(square_metab[np.ix_(order, order)])).statistic)
null = np.array(null)
p_value = (np.sum(np.abs(null) >= abs(observed)) + 1) / (len(null) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].scatter(d_prot, d_metab, s=5, alpha=0.25, color="#4C72B0")
axes[0].set(xlabel="proteomic distance between two patients",
            ylabel="metabolomic distance",
            title=f"Spearman rho = {observed:.3f}")
axes[1].hist(null, bins=40, color="#9AA5B1", label="permuted null")
axes[1].axvline(observed, color="#C44E52", lw=2, label="observed")
axes[1].set(xlabel="Spearman rho", ylabel="count",
            title=f"Mantel test: p = {p_value:.3f}")
axes[1].legend()
fig.tight_layout()

🧬 **How to read this.** A clearly positive, significant correlation means the two layers
describe overlapping biology, and integration should reinforce a shared signal. A
correlation near zero means the layers are largely independent — integration then adds
*new* information rather than confirmation, and you should expect the joint model to find
factors specific to one view.

Either answer is useful. What would be a mistake is not asking.

## 3. Similarity Network Fusion

SNF ([Wang *et al.*, Nat Methods 2014](https://www.nature.com/articles/nmeth.2810)) starts
from a simple idea: **stop comparing features, compare patients.**

1. For each view, build a **patient × patient similarity network** — one node per patient,
   edge weights from a scaled distance, sparsified to each patient's *K* nearest
   neighbours. Feature counts and units disappear at this step.
2. **Fuse** the networks by iterative diffusion: each network is updated using the
   others' local structure, so similarities supported by both layers get reinforced and
   similarities supported by only one get damped.
3. Cluster the fused network.

The two hyperparameters are *K*, the neighbourhood size (typically 10–30, and it must be
well below your sample count), and *µ*, a scaling factor for the similarity kernel
(0.3–0.8). With 45 patients we take K = 15.

In [ ]:
from snf import make_affinity, snf

K, MU, T = 15, 0.5, 20

affinities = make_affinity(
    [proteomics.values, metabolomics.values], metric="euclidean", K=K, mu=MU
)
fused = snf(affinities, K=K, t=T)
print("affinity matrices:", [a.shape for a in affinities], "-> fused:", fused.shape)

In [ ]:
labels_in_order = groups.tolist()
sort_index = np.argsort([GROUP_ORDER.index(g) for g in labels_in_order])
titles = ["Proteomics similarity", "Metabolomics similarity", "Fused network"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, matrix, title in zip(axes, affinities + [fused], titles):
    ordered = matrix[np.ix_(sort_index, sort_index)]
    im = ax.imshow(ordered, cmap="magma")
    boundaries = np.cumsum([sum(1 for g in labels_in_order if g == grp) for grp in GROUP_ORDER])[:-1]
    for b in boundaries:
        ax.axhline(b - 0.5, color="white", lw=1)
        ax.axvline(b - 0.5, color="white", lw=1)
    ax.set(title=title, xticks=[], yticks=[])
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Patients ordered Con | CSKP | CRKP — block structure would mean the groups differ", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "snf_affinity_matrices.png", bbox_inches="tight", dpi=200)

Look for **blocks along the diagonal**. Sharp blocks aligned with the white dividers would
mean patients of the same group are more similar to each other than to other groups. A
smooth matrix means the dominant similarity structure is something else — most likely how
severely ill each patient is, which cuts across the microbiological groups.

### Clustering the fused network

SNF's companion step is spectral clustering, which works directly on a similarity matrix
(no need to invent a distance in feature space). `snfpy` can also estimate how many
clusters the fused network supports, using the eigengap heuristic.

In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

try:
    from snf import get_n_clusters

    estimated = get_n_clusters(fused)
    print("cluster counts suggested by the eigengap heuristic:", estimated)
except Exception as error:
    print("could not run get_n_clusters:", error)

N_CLUSTERS = 3
clustering = SpectralClustering(
    n_clusters=N_CLUSTERS, affinity="precomputed", random_state=42, assign_labels="kmeans"
)
cluster_labels = clustering.fit_predict(fused)

results = pd.DataFrame({"group": groups.values, "snf_cluster": cluster_labels}, index=shared)
contingency = pd.crosstab(results["group"], results["snf_cluster"]).reindex(GROUP_ORDER)
print(f"\nNMI (cluster vs group) : {normalized_mutual_info_score(groups, cluster_labels):.3f}")
print(f"ARI (cluster vs group) : {adjusted_rand_score(groups, cluster_labels):.3f}")
contingency

**How to read the numbers.** Normalised mutual information and the adjusted Rand index
both compare our unsupervised clusters with the clinical groups. Both are 0 when the
clusters tell you nothing about the groups beyond chance, and 1 when they are identical.
In practice, on a cohort like this, expect something modest.

- **Near 0** — the strongest structure in the combined data is *not* the CRKP/CSKP/Con
  distinction. That is a real result, not a failure: it says the serum profile of a septic
  patient is dominated by how ill they are, and the pathogen's resistance status is a small
  perturbation on top. It is also why the published biomarker panel needed **supervised**
  machine learning — you have to *ask* for the distinction to see it.
- **Clearly above 0** — some of the dominant structure does track the groups, and it is
  worth asking which patients the clusters put together and whether the clusters split one
  group in two (a possible severity gradient within a group).

Either way, the next cell is the important one: find out what the clusters track, instead
of only whether they track the label you hoped for.

In [ ]:
# What do the clusters actually track, if not the groups?
results = results.join(metadata.loc[shared, ["age", "c_reactive_protein", "procalcitonin",
                                             "creatinine", "platelet_count",
                                             "neutrophil_lymphocyte_ratio", "sex"]])
cluster_profile = results.groupby("snf_cluster")[
    ["age", "c_reactive_protein", "procalcitonin", "creatinine", "platelet_count",
     "neutrophil_lymphocyte_ratio"]
].median()
cluster_profile.insert(0, "n", results["snf_cluster"].value_counts().sort_index())
cluster_profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, variable in zip(axes, ["procalcitonin", "creatinine", "platelet_count"]):
    sns.boxplot(data=results, x="snf_cluster", y=variable, ax=ax, showfliers=False,
                hue="snf_cluster", palette="Blues", legend=False)
    sns.stripplot(data=results, x="snf_cluster", y=variable, ax=ax, color="black", size=3, alpha=0.6)
    pvalue = stats.kruskal(*[g[variable].dropna() for _, g in results.groupby("snf_cluster")]).pvalue
    ax.set(title=f"{variable}  (Kruskal-Wallis p = {pvalue:.3f})")
fig.suptitle("Clinical profile of the SNF clusters", y=1.04)
fig.tight_layout()
fig.savefig(OUT_DIR / "snf_cluster_clinical_profile.png", bbox_inches="tight", dpi=200)

## 4. Multi-Omics Factor Analysis (MOFA)

SNF told us about patients. MOFA tells us about **the sources of variation themselves**.

**Multi-Omics Factor Analysis (MOFA)** ([Argelaguet *et al.* 2018](https://www.embopress.org/doi/full/10.15252/msb.20178124)) is a statistical framework designed to integrate heterogeneous molecular datasets—such as transcriptomics, proteomics, and metabolomics—measured from the same set of biological samples. Think of it as an unsupervised dimensionality reduction tool that finds hidden common denominators across multiple biological layers.

**The Core Intuition: Matrix Factorization**
MOFA extends standard matrix factorization techniques to multi-view data. For each omics data type, the method takes a high-dimensional matrix of samples by features and decomposes it into two lower-dimensional matrices:

* **Latent Factors ($Z$):** Represent hidden biological drivers, such as patient subgroups, disease progression stages, or environmental responses. These factors capture the shared or unique variation across the samples.
* **Feature Weights ($W$):** Quantify how strongly each individual gene, protein, or metabolite contributes to each factor, allowing you to link latent signals back to specific biological pathways.

**How MOFA Handles Multi-Omics Complexity**
The true power of MOFA lies in its ability to model both shared and specific signals across diverse data types through Bayesian group sparsity:

* **Global Factors:** Capture variation present across multiple omics layers, revealing coordinated biological responses (e.g., a transcriptional change that directly alters protein abundance).
* **Omics-Specific Factors:** Isolate variation unique to a single modality (e.g., metabolic shifts not reflected in gene expression), preventing data integration from washing out single-source signals.
* **Missing Data Resilience:** Naturally accommodates missing measurements in specific omics layers for certain samples, estimating latent factors using whatever data is available.

**Practical Outputs and Use Cases**
MOFA translates complex multi-layered datasets into an interpretable low-dimensional space ideal for downstream tasks:

* **Patient Stratification:** Clustering samples based on latent factors to discover clinically relevant disease subtypes.
* **Biomarker Discovery:** Inspecting feature weights to pinpoint key multi-omics drivers.
* **Data Visualization:** Projecting multi-modal samples into 2D or 3D scatter plots for intuitive exploration.

Sparsity priors (`spikeslab_weights`, `ard_weights`) push most weights to zero, so each
factor ends up associated with a short, interpretable list of features.

**`spikeslab_weights`**

Function: Forces the weights of non-informative features down to exactly zero, ensuring each latent factor is defined by a sparse, highly interpretable subset of genes, proteins, or metabolites rather than noise from every measured feature.  

Configuration: Set to True (the default in MOFA+) for standard multi-omics integrations where downstream biological interpretation is critical. Set to False only if you require dense representations akin to standard PCA. 

**`ard_weights`** (Automatic Relevance Determination Weights)

Function: Automatically shrinks entire data modalities to zero for a given factor if that omics layer does not contribute meaningful variance. This allows the model to cleanly distinguish between global multi-omics factors and single-view specific drivers.

Configuration: Set to True (recommended and default when handling multiple views) to let the model dynamically learn cross-modal architecture. Disable only if you need to force every factor to use all data modalities uniformly.

In [ ]:
try:
    from mofapy2.run.entry_point import entry_point

    MOFA_AVAILABLE = True
except Exception as error:
    MOFA_AVAILABLE = False
    print("mofapy2 is not available in this runtime:", error)
    print("Read on — the interpretation section explains what the output looks like.")

`mofapy2` wants the data as a nested list, `[view][group]`, each entry a
**samples × features** array. We have one group of samples, so each view is a list of one.

In [ ]:
if MOFA_AVAILABLE:
    view_names = ["proteomics", "metabolomics"]
    data_matrices = [[proteomics.values], [metabolomics.values]]
    feature_names = [proteomics.columns.tolist(), metabolomics.columns.tolist()]

    model = entry_point()
    model.set_data_options(scale_views=True, center_groups=True)
    model.set_data_matrix(
        data_matrices,
        views_names=view_names,
        groups_names=["all_patients"],
        samples_names=[shared],
        features_names=feature_names,
        likelihoods=["gaussian", "gaussian"],
    )
    model.set_model_options(factors=8, spikeslab_weights=True, ard_weights=True)
    model.set_train_options(convergence_mode="fast", dropR2=0.01, gpu_mode=False,
                           seed=42, verbose=False)
    model.build()
    model.run()
    model.save(outfile=str(OUT_DIR / "mofa_model.hdf5"), save_data=True)
    print("model saved to", OUT_DIR / "mofa_model.hdf5")

### Variance explained: the first plot to look at

In [ ]:
if MOFA_AVAILABLE:
    import mofax

    mofa = mofax.mofa_model(str(OUT_DIR / "mofa_model.hdf5"))
    # long table with columns Factor / View / Group / R2
    r2 = mofa.get_variance_explained()
    r2_matrix = r2.pivot_table(index="Factor", columns="View", values="R2")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(r2_matrix, annot=True, fmt=".1f", cmap="Blues", ax=axes[0],
                cbar_kws={"label": "variance explained (%)"})
    axes[0].set(title="Variance explained per factor and view")
    r2_matrix.sum(axis=1).plot(kind="bar", ax=axes[1], color="#4C72B0")
    axes[1].set(ylabel="total variance explained (%)", title="Factor importance")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "mofa_variance_explained.png", bbox_inches="tight", dpi=200)
    display(r2_matrix.round(2))

### Where do the patients sit?

In [ ]:
if MOFA_AVAILABLE:
    factors = mofa.get_factors(df=True)
    factors.index = shared if len(factors) == len(shared) else factors.index
    factors = factors.join(metadata.loc[shared, ["group", "age", "sex", "c_reactive_protein",
                                                 "procalcitonin", "creatinine"]])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for group in GROUP_ORDER:
        sub = factors[factors["group"] == group]
        axes[0].scatter(sub["Factor1"], sub["Factor2"], label=group, s=70,
                        color=GROUP_COLOURS[group], edgecolor="white")
    axes[0].set(xlabel="Factor 1", ylabel="Factor 2", title="Patients in factor space")
    axes[0].legend(title="group")

    factor_cols = [c for c in factors.columns if c.startswith("Factor")]
    long = factors.melt(id_vars="group", value_vars=factor_cols[:4],
                        var_name="factor", value_name="score")
    sns.boxplot(data=long, x="factor", y="score", hue="group", hue_order=GROUP_ORDER,
                palette=GROUP_COLOURS, ax=axes[1], showfliers=False)
    axes[1].set(title="Factor scores by group")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "mofa_factors.png", bbox_inches="tight", dpi=200)

### Which factors are biology, and which are noise?

A factor is just a direction of variance; MOFA does not know what it means. The way to
find out is to ask what it correlates with: our groups, the clinical variables, or nothing
we can name.

In [ ]:
if MOFA_AVAILABLE:
    clinical_vars = ["age", "c_reactive_protein", "procalcitonin", "creatinine"]
    rows = []
    for factor in factor_cols:
        row = {"factor": factor}
        kruskal = stats.kruskal(*[factors.loc[factors["group"] == g, factor] for g in GROUP_ORDER])
        row["p (group difference)"] = kruskal.pvalue
        for variable in clinical_vars:
            valid = factors[[factor, variable]].dropna()
            row[f"rho {variable}"] = stats.spearmanr(valid[factor], valid[variable]).statistic
        rows.append(row)
    factor_meaning = pd.DataFrame(rows).set_index("factor")
    display(factor_meaning.round(3))

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(factor_meaning) + 2))
    sns.heatmap(factor_meaning[[c for c in factor_meaning.columns if c.startswith("rho")]],
                annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
    ax.set(title="What each MOFA factor tracks")
    fig.tight_layout()

### The features behind a factor

Finally, the interpretable part: for a factor of interest, which proteins and metabolites
load on it most strongly? Those are the candidate members of a cross-layer module — and
the input to Multi-omics II.

In [ ]:
if MOFA_AVAILABLE:
    weights = mofa.get_weights(df=True)
    interesting = factor_meaning["p (group difference)"].idxmin()
    print("factor most associated with the clinical groups:", interesting)

    top_features = {}
    for view_name, view in views.items():
        view_weights = weights.loc[weights.index.isin(view.columns), interesting]
        top = view_weights.reindex(view_weights.abs().sort_values(ascending=False).index).head(12)
        if view_name == "proteomics":
            top.index = [
                protein_info.loc[i, "genes"] if i in protein_info.index
                and pd.notna(protein_info.loc[i, "genes"]) else i
                for i in top.index
            ]
        top_features[view_name] = top

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
    for ax, (view_name, top) in zip(axes, top_features.items()):
        colours = ["#C44E52" if w > 0 else "#4C72B0" for w in top.values]
        ax.barh(top.index.astype(str)[::-1], top.values[::-1], color=colours[::-1])
        ax.axvline(0, color="grey", lw=0.8)
        ax.set(xlabel=f"weight on {interesting}", title=view_name)
    fig.suptitle(f"Top features loading on {interesting}", y=1.03)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "mofa_top_weights.png", bbox_inches="tight", dpi=200)

## 5. Save the results

In [ ]:
proteomics.to_csv(OUT_DIR / "view_proteomics_zscored.csv")
metabolomics.to_csv(OUT_DIR / "view_metabolomics_zscored.csv")
results.to_csv(OUT_DIR / "snf_clusters.csv")
if MOFA_AVAILABLE:
    factors.to_csv(OUT_DIR / "mofa_factors.csv")
    weights.to_csv(OUT_DIR / "mofa_weights.csv")
!ls -lh multiomics/results

## 📚 Further reading

- Wang B *et al.* (2014) *Similarity network fusion for aggregating data types on a genomic
  scale.* Nat Methods 11:333–337.
- Argelaguet R *et al.* (2018) *Multi-Omics Factor Analysis — a framework for unsupervised
  integration of multi-omics data sets.* Mol Syst Biol 14:e8124.
- Argelaguet R *et al.* (2020) *MOFA+: a statistical framework for comprehensive
  integration of multi-modal single-cell data.* Genome Biol 21:111.
- Baião AR *et al.* (2025) *A technical review of multi-omics data integration methods:
  from classical statistical to deep generative approaches.* Brief Bioinform 26:bbaf355.
- Cantini L *et al.* (2021) *Benchmarking joint multi-omics dimensionality reduction
  approaches for the study of cancer.* Nat Commun 12:124. — read before you trust any single method.